In [2]:
# -*- coding: utf-8 -*-
"""Bước 1: tự tính khoảng cách và tự tìm láng giềng, chưa dùng thư viện.

Chú thích viết tiếng Việt có dấu, phần in ra màn hình viết không dấu.
"""

import numpy as np
import pandas as pd

# Ba bạn: A và C học gần như y hệt nhau, B học khác hẳn
ba_ban = {
    "A": np.array([5.0, 3.0, 4000.0]),
    "B": np.array([25.0, 9.0, 4100.0]),
    "C": np.array([6.0, 3.5, 7000.0]),
}


def kc(u, v):
    """Khoang cach Euclid giua hai diem."""
    return np.sqrt(((u - v) ** 2).sum())


print("Khoang cach KHI CHUA chuan hoa")
print(f"  d(A, B) = {kc(ba_ban['A'], ba_ban['B']):8.1f}")
print(f"  d(A, C) = {kc(ba_ban['A'], ba_ban['C']):8.1f}")
print()

# Chuẩn hoá bằng trung bình và độ lệch chuẩn của cả bộ dữ liệu
df = pd.read_csv("data/sinh_vien_mo_rong.csv")
X = df[["gio_on", "diem_giua_ky", "chi_tieu"]].to_numpy(dtype=float)
tb = X.mean(axis=0)
dl = X.std(axis=0)
print("Do lech chuan cua ba cot:")
print(f"  gio_on = {dl[0]:.2f}   diem_giua_ky = {dl[1]:.2f}"
      f"   chi_tieu = {dl[2]:.0f}")
print()

chuan = {ten: (v - tb) / dl for ten, v in ba_ban.items()}
print("Khoang cach SAU KHI chuan hoa")
print(f"  d(A, B) = {kc(chuan['A'], chuan['B']):.2f}")
print(f"  d(A, C) = {kc(chuan['A'], chuan['C']):.2f}")
print()

# Tìm 5 láng giềng gần nhất của một bạn mới, chỉ dùng hai cột học tập
X2 = df[["gio_on", "diem_giua_ky"]].to_numpy(dtype=float)
y = df["qua_mon"].to_numpy()
tb2, dl2 = X2.mean(axis=0), X2.std(axis=0)
ban_moi = np.array([12.0, 6.5])
X2c = (X2 - tb2) / dl2
moi_c = (ban_moi - tb2) / dl2
khoang = np.sqrt(((X2c - moi_c) ** 2).sum(axis=1))
gan = np.argsort(khoang)[:5]

print("Ban moi: on 12 gio, diem giua ky 6.5")
print("Nam lang gieng gan nhat:")
for i in gan:
    ket = "qua" if y[i] == 1 else "rot"
    print(f"  gio {X2[i, 0]:5.1f}  diem {X2[i, 1]:4.1f}  "
          f"khoang cach {khoang[i]:.3f}  -> {ket}")

so_qua = int(y[gan].sum())
print(f"Bo phieu: {so_qua} qua, {5 - so_qua} rot")
print("Du doan:", "qua mon" if so_qua >= 3 else "rot mon")

Khoang cach KHI CHUA chuan hoa
  d(A, B) =    102.2
  d(A, C) =   3000.0

Do lech chuan cua ba cot:
  gio_on = 7.82   diem_giua_ky = 2.09   chi_tieu = 1692

Khoang cach SAU KHI chuan hoa
  d(A, B) = 3.85
  d(A, C) = 1.79

Ban moi: on 12 gio, diem giua ky 6.5
Nam lang gieng gan nhat:
  gio  10.5  diem  6.4  khoang cach 0.198  -> qua
  gio  12.5  diem  6.1  khoang cach 0.202  -> qua
  gio  14.0  diem  6.4  khoang cach 0.260  -> qua
  gio   9.5  diem  6.1  khoang cach 0.373  -> rot
  gio  10.5  diem  5.8  khoang cach 0.386  -> qua
Bo phieu: 4 qua, 1 rot
Du doan: qua mon


In [3]:
# -*- coding: utf-8 -*-
"""Bước 2: đo hậu quả của việc quên chuẩn hoá trên tập kiểm tra."""

import pandas as pd
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

df = pd.read_csv("data/sinh_vien_mo_rong.csv")
y = df["qua_mon"]
X3 = df[["gio_on", "diem_giua_ky", "chi_tieu"]]

X_tr, X_te, y_tr, y_te = train_test_split(
    X3, y, test_size=0.25, random_state=17, stratify=y
)


def cham(mo_hinh, cot):
    mo_hinh.fit(X_tr[cot], y_tr)
    return accuracy_score(y_te, mo_hinh.predict(X_te[cot]))


def knn_chuan():
    """KNN co chuan hoa, gop lam mot khoi bang pipeline."""
    return make_pipeline(StandardScaler(), KNeighborsClassifier(7))


ba_cot = ["gio_on", "diem_giua_ky", "chi_tieu"]
hai_cot = ["gio_on", "diem_giua_ky"]

a = cham(KNeighborsClassifier(n_neighbors=7), ba_cot)
b = cham(knn_chuan(), ba_cot)
c = cham(knn_chuan(), hai_cot)

print(f"Doan bua ca lop qua            : {y_te.mean():.4f}")
print(f"Ba cot, khong chuan hoa        : {a:.4f}")
print(f"Ba cot, co chuan hoa           : {b:.4f}")
print(f"Bo cot chi_tieu, co chuan hoa  : {c:.4f}")

Doan bua ca lop qua            : 0.6000
Ba cot, khong chuan hoa        : 0.5333
Ba cot, co chuan hoa           : 0.8667
Bo cot chi_tieu, co chuan hoa  : 0.9000


In [4]:
# -*- coding: utf-8 -*-
"""Bước 3: chọn số láng giềng k bằng kiểm định chéo."""

import pandas as pd
from sklearn.metrics import accuracy_score
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.neighbors import KNeighborsClassifier

df = pd.read_csv("data/hai_vang_trang.csv")
X = df[["x1", "x2"]]
y = df["nhan"]

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y
)

# Hai cột x1 và x2 có thang đo gần nhau, độ lệch chuẩn chỉ chênh khoảng
# 1.6 lần, nên ở bộ này không chuẩn hoá cũng được.
print("Accuracy tren tap hoc va tap kiem tra theo k")
print("   k   tap hoc   tap kiem tra")
for k in [1, 5, 13, 31, 101]:
    mh = KNeighborsClassifier(n_neighbors=k).fit(X_tr, y_tr)
    a_hoc = accuracy_score(y_tr, mh.predict(X_tr))
    a_kt = accuracy_score(y_te, mh.predict(X_te))
    print(f" {k:3d}    {a_hoc:.4f}         {a_kt:.4f}")
print()

# Chọn k chỉ dựa trên tập học, không nhìn tập kiểm tra
print("Kiem dinh cheo 5 phan tren tap hoc:")
diem = {}
for k in range(1, 52, 2):
    diem[k] = cross_val_score(
        KNeighborsClassifier(n_neighbors=k), X_tr, y_tr, cv=5
    ).mean()
k_tot = max(diem, key=diem.get)

for k in [1, 5, 9, 13, 17, 25, 51]:
    dau = " <- tot nhat" if k == k_tot else ""
    print(f"  k = {k:2d}   diem trung binh {diem[k]:.4f}{dau}")
print()

# Chọn xong mới mở tập kiểm tra, và chỉ mở một lần
mh = KNeighborsClassifier(n_neighbors=k_tot).fit(X_tr, y_tr)
print(f"k tot nhat = {k_tot}")
acc = accuracy_score(y_te, mh.predict(X_te))
print(f"Accuracy tren tap kiem tra = {acc:.4f}")

Accuracy tren tap hoc va tap kiem tra theo k
   k   tap hoc   tap kiem tra
   1    1.0000         0.8000
   5    0.8756         0.9067
  13    0.8844         0.9200
  31    0.8667         0.8933
 101    0.8311         0.8533

Kiem dinh cheo 5 phan tren tap hoc:
  k =  1   diem trung binh 0.7778
  k =  5   diem trung binh 0.8444
  k =  9   diem trung binh 0.8444
  k = 13   diem trung binh 0.8667 <- tot nhat
  k = 17   diem trung binh 0.8533
  k = 25   diem trung binh 0.8622
  k = 51   diem trung binh 0.8533

k tot nhat = 13
Accuracy tren tap kiem tra = 0.9200


In [5]:
# -*- coding: utf-8 -*-
"""Bước 4: đặt KNN cạnh hai mô hình đã học trên dữ liệu có biên cong."""

import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier

df = pd.read_csv("data/hai_vang_trang.csv")
X = df[["x1", "x2"]]
y = df["nhan"]

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y
)

cac_mo_hinh = [
    ("Hoi quy logistic", LogisticRegression()),
    ("Naive Bayes     ", GaussianNB()),
    ("KNN, k = 13     ", KNeighborsClassifier(n_neighbors=13)),
]

print("Accuracy tren tap kiem tra, bo hai vang trang")
for ten, mh in cac_mo_hinh:
    mh.fit(X_tr, y_tr)
    print(f"  {ten}: {accuracy_score(y_te, mh.predict(X_te)):.4f}")

Accuracy tren tap kiem tra, bo hai vang trang
  Hoi quy logistic: 0.8533
  Naive Bayes     : 0.8533
  KNN, k = 13     : 0.9200


In [6]:
# -*- coding: utf-8 -*-
"""Bước 5: KNN cho bài toán hồi quy, trên dữ liệu giá nhà của bài 1."""

import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsRegressor

df = pd.read_csv("data/gia_nha.csv")
X = df[["dien_tich"]]
y = df["gia"]

# Cùng cách chia với bài 1 để so sánh công bằng
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.2, random_state=42
)

knn = KNeighborsRegressor(n_neighbors=5).fit(X_tr, y_tr)
duong = LinearRegression().fit(X_tr, y_tr)

print("MAE tren tap kiem tra (ty dong)")
mae_knn = mean_absolute_error(y_te, knn.predict(X_te))
mae_duong = mean_absolute_error(y_te, duong.predict(X_te))
print(f"  KNN, k = 5 : {mae_knn:.4f}")
print(f"  Duong thang: {mae_duong:.4f}")
print()

can_moi = pd.DataFrame({"dien_tich": [60.0, 100.0, 150.0, 250.0]})
print("Du doan cho bon can moi (ty dong)")
print("  dien tich     KNN   duong thang")
for dt, a, b in zip(
    can_moi["dien_tich"], knn.predict(can_moi), duong.predict(can_moi)
):
    print(f"   {dt:6.0f} m2  {a:6.2f}        {b:6.2f}")
print()

# Năm láng giềng của căn 250 m2 chính là những căn rộng nhất đã thấy
_, chi_so = knn.kneighbors(pd.DataFrame({"dien_tich": [250.0]}))
lg = X_tr.iloc[chi_so[0]]["dien_tich"].tolist()
print("Nam lang gieng cua can 250 m2:", ", ".join(f"{v:.1f}" for v in lg))

MAE tren tap kiem tra (ty dong)
  KNN, k = 5 : 0.4030
  Duong thang: 0.2578

Du doan cho bon can moi (ty dong)
  dien tich     KNN   duong thang
       60 m2    4.95          5.12
      100 m2    8.32          8.24
      150 m2    9.32         12.13
      250 m2    9.32         19.91

Nam lang gieng cua can 250 m2: 117.5, 115.9, 111.7, 111.2, 104.2


In [7]:
# -*- coding: utf-8 -*-
"""Bài tập 1: Tự viết KNN không dùng thư viện và so sánh với scikit-learn."""

import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier


def knn_du_doan(X_hoc, y_hoc, x_moi, k):
    """Dự đoán nhãn cho một mẫu mới x_moi dựa trên k láng giềng gần nhất."""
    # Bước 1: Tính khoảng cách Euclid từ x_moi tới mọi điểm trong X_hoc
    khoang_cach = np.sqrt(((X_hoc - x_moi) ** 2).sum(axis=1))

    # Bước 2: Lấy chỉ số của k khoảng cách nhỏ nhất bằng np.argsort
    chi_so_gan = np.argsort(khoang_cach)[:k]

    # Bước 3: Cho k láng giềng bỏ phiếu (nhãn 0 hoặc 1)
    so_phieu_lop_1 = int(y_hoc[chi_so_gan].sum())
    return 1 if so_phieu_lop_1 > (k / 2) else 0


# Đọc dữ liệu hai vầng trăng và chia tập học / tập kiểm tra giống mục 4
df = pd.read_csv("data/hai_vang_trang.csv")
X = df[["x1", "x2"]].to_numpy(dtype=float)
y = df["nhan"].to_numpy()

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y
)

k = 13

# Dự đoán cho toàn bộ tập kiểm tra bằng hàm tự viết
y_du_doan_tu_viet = np.array([knn_du_doan(X_tr, y_tr, x_moi, k) for x_moi in X_te])
acc_tu_viet = accuracy_score(y_te, y_du_doan_tu_viet)

# Dự đoán bằng KNeighborsClassifier của thư viện scikit-learn
mo_hinh_sklearn = KNeighborsClassifier(n_neighbors=k).fit(X_tr, y_tr)
acc_sklearn = accuracy_score(y_te, mo_hinh_sklearn.predict(X_te))

print(f"Ket qua voi k = {k} tren tap kiem tra bo hai vang trang:")
print(f"  Accuracy KNN tu viet        : {acc_tu_viet:.4f}")
print(f"  Accuracy KNeighborsClassifier: {acc_sklearn:.4f}")
print(f"  Hai ket qua bang nhau       : {'Co' if acc_tu_viet == acc_sklearn else 'Khong'}")

Ket qua voi k = 13 tren tap kiem tra bo hai vang trang:
  Accuracy KNN tu viet        : 0.9200
  Accuracy KNeighborsClassifier: 0.9200
  Hai ket qua bang nhau       : Co


In [8]:
# -*- coding: utf-8 -*-
"""Bài tập 2: Chạy lại e3_chon_k.py với khoảng cách Manhattan."""

import pandas as pd
from sklearn.metrics import accuracy_score
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.neighbors import KNeighborsClassifier

df = pd.read_csv("data/hai_vang_trang.csv")
X = df[["x1", "x2"]]
y = df["nhan"]

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y
)

print("Accuracy tren tap hoc va tap kiem tra theo k (Manhattan)")
print("   k   tap hoc   tap kiem tra")
for k in [1, 5, 13, 31, 101]:
    mh = KNeighborsClassifier(
        n_neighbors=k, metric="manhattan"
    ).fit(X_tr, y_tr)
    a_hoc = accuracy_score(y_tr, mh.predict(X_tr))
    a_kt = accuracy_score(y_te, mh.predict(X_te))
    print(f" {k:3d}    {a_hoc:.4f}         {a_kt:.4f}")
print()

# Chọn k bằng kiểm định chéo 5 phần trên tập học với khoảng cách Manhattan
print("Kiem dinh cheo 5 phan tren tap hoc (Manhattan):")
diem = {}
for k in range(1, 52, 2):
    mh_cv = KNeighborsClassifier(n_neighbors=k, metric="manhattan")
    diem[k] = cross_val_score(mh_cv, X_tr, y_tr, cv=5).mean()

k_tot = max(diem, key=diem.get)

# In các mốc k tiêu biểu và k_tot
cac_moc_in = sorted(set([1, 5, 9, 13, 17, 25, 51, k_tot]))
for k in cac_moc_in:
    dau = " <- tot nhat" if k == k_tot else ""
    print(f"  k = {k:2d}   diem trung binh {diem[k]:.4f}{dau}")
print()

# Đánh giá trên tập kiểm tra với k tốt nhất của Manhattan
mh_manhattan = KNeighborsClassifier(
    n_neighbors=k_tot, metric="manhattan"
).fit(X_tr, y_tr)
acc_manhattan = accuracy_score(y_te, mh_manhattan.predict(X_te))

print("So sanh Manhattan va Euclid:")
print(f"  Manhattan: k tot nhat = {k_tot:2d}, CV = {diem[k_tot]:.4f}, Accuracy kiem tra = {acc_manhattan:.4f}")
print(f"  Euclid   : k tot nhat = 13, CV = 0.8667, Accuracy kiem tra = 0.9200")
print("Nhan xet: Doi cach do khoang cach khong lam ket qua khac di nhieu (0.9067 so voi 0.9200).")

Accuracy tren tap hoc va tap kiem tra theo k (Manhattan)
   k   tap hoc   tap kiem tra
   1    1.0000         0.7867
   5    0.8756         0.8933
  13    0.8667         0.9067
  31    0.8711         0.9067
 101    0.8311         0.8400

Kiem dinh cheo 5 phan tren tap hoc (Manhattan):
  k =  1   diem trung binh 0.7822
  k =  5   diem trung binh 0.8444
  k =  9   diem trung binh 0.8222
  k = 13   diem trung binh 0.8489
  k = 17   diem trung binh 0.8622
  k = 25   diem trung binh 0.8578
  k = 27   diem trung binh 0.8667 <- tot nhat
  k = 51   diem trung binh 0.8489

So sanh Manhattan va Euclid:
  Manhattan: k tot nhat = 27, CV = 0.8667, Accuracy kiem tra = 0.9067
  Euclid   : k tot nhat = 13, CV = 0.8667, Accuracy kiem tra = 0.9200
Nhan xet: Doi cach do khoang cach khong lam ket qua khac di nhieu (0.9067 so voi 0.9200).


In [9]:
# -*- coding: utf-8 -*-
"""Bài tập 3: Đổi đơn vị cột chi_tieu từ nghìn đồng sang triệu đồng."""

import pandas as pd
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

df = pd.read_csv("data/sinh_vien_mo_rong.csv")
y = df["qua_mon"]

# Bản gốc (chi_tieu tính bằng nghìn đồng)
X_nghin = df[["gio_on", "diem_giua_ky", "chi_tieu"]].copy()

# Bản đổi đơn vị (chi_tieu tính bằng triệu đồng = chia cho 1000)
X_trieu = df[["gio_on", "diem_giua_ky", "chi_tieu"]].copy()
X_trieu["chi_tieu"] = X_trieu["chi_tieu"] / 1000.0

# Chia tập học và tập kiểm tra với cùng random_state=17 và stratify=y
X_tr_n, X_te_n, y_tr, y_te = train_test_split(
    X_nghin, y, test_size=0.25, random_state=17, stratify=y
)
X_tr_t, X_te_t, _, _ = train_test_split(
    X_trieu, y, test_size=0.25, random_state=17, stratify=y
)

# 1. Không chuẩn hoá
mh_khong_chuan_nghin = KNeighborsClassifier(n_neighbors=7).fit(X_tr_n, y_tr)
acc_khong_chuan_nghin = accuracy_score(y_te, mh_khong_chuan_nghin.predict(X_te_n))

mh_khong_chuan_trieu = KNeighborsClassifier(n_neighbors=7).fit(X_tr_t, y_tr)
acc_khong_chuan_trieu = accuracy_score(y_te, mh_khong_chuan_trieu.predict(X_te_t))

# 2. Có chuẩn hoá bằng StandardScaler trong make_pipeline
mh_co_chuan_nghin = make_pipeline(
    StandardScaler(), KNeighborsClassifier(n_neighbors=7)
).fit(X_tr_n, y_tr)
acc_co_chuan_nghin = accuracy_score(y_te, mh_co_chuan_nghin.predict(X_te_n))

mh_co_chuan_trieu = make_pipeline(
    StandardScaler(), KNeighborsClassifier(n_neighbors=7)
).fit(X_tr_t, y_tr)
acc_co_chuan_trieu = accuracy_score(y_te, mh_co_chuan_trieu.predict(X_te_t))

dl = X_trieu.to_numpy(dtype=float).std(axis=0)
print("Do lech chuan ba cot SAU KHI doi chi_tieu sang trieu dong:")
print(f"  gio_on = {dl[0]:.2f}   diem_giua_ky = {dl[1]:.2f}   chi_tieu = {dl[2]:.2f}")
print()
print("So sanh Accuracy tren tap kiem tra (k = 7):")
print(f"  1. Khong chuan hoa (nghin dong - muc 3): {acc_khong_chuan_nghin:.4f}")
print(f"  2. Khong chuan hoa (trieu dong - moi)  : {acc_khong_chuan_trieu:.4f}")
print(f"  3. Co chuan hoa    (nghin dong - muc 3): {acc_co_chuan_nghin:.4f}")
print(f"  4. Co chuan hoa    (trieu dong - moi)  : {acc_co_chuan_trieu:.4f}")

Do lech chuan ba cot SAU KHI doi chi_tieu sang trieu dong:
  gio_on = 7.82   diem_giua_ky = 2.09   chi_tieu = 1.69

So sanh Accuracy tren tap kiem tra (k = 7):
  1. Khong chuan hoa (nghin dong - muc 3): 0.5333
  2. Khong chuan hoa (trieu dong - moi)  : 0.8667
  3. Co chuan hoa    (nghin dong - muc 3): 0.8667
  4. Co chuan hoa    (trieu dong - moi)  : 0.8667


In [10]:
# -*- coding: utf-8 -*-
"""Bài tập 4: So sánh KNN bỏ phiếu đều và bỏ phiếu có trọng số khoảng cách."""

import pandas as pd
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier

df = pd.read_csv("data/hai_vang_trang.csv")
X = df[["x1", "x2"]]
y = df["nhan"]

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y
)

print("So sanh weights='uniform' (muc 4) va weights='distance' tren bo hai vang trang:")
print("-" * 74)
print(f"{'k':>4} | {'Tap hoc (deu)':>14} {'Tap KT (deu)':>13} | {'Tap hoc (distance)':>19} {'Tap KT (distance)':>18}")
print("-" * 74)

for k in [1, 13, 101]:
    # Bỏ phiếu đều như ở mục 4 (weights='uniform')
    mh_deu = KNeighborsClassifier(n_neighbors=k, weights="uniform").fit(X_tr, y_tr)
    hoc_deu = accuracy_score(y_tr, mh_deu.predict(X_tr))
    kt_deu = accuracy_score(y_te, mh_deu.predict(X_te))

    # Láng giềng có trọng số theo khoảng cách (weights='distance')
    mh_kc = KNeighborsClassifier(n_neighbors=k, weights="distance").fit(X_tr, y_tr)
    hoc_kc = accuracy_score(y_tr, mh_kc.predict(X_tr))
    kt_kc = accuracy_score(y_te, mh_kc.predict(X_te))

    print(f"{k:4d} | {hoc_deu:14.4f} {kt_deu:13.4f} | {hoc_kc:19.4f} {kt_kc:18.4f}")

print("-" * 74)
print("Ket luan voi k = 101: Trong so khoang cach giup Accuracy tap kiem tra")
print("tang tu 0.8533 len 0.9067, giup mo hinh bot cau tha ro ret.")

So sanh weights='uniform' (muc 4) va weights='distance' tren bo hai vang trang:
--------------------------------------------------------------------------
   k |  Tap hoc (deu)  Tap KT (deu) |  Tap hoc (distance)  Tap KT (distance)
--------------------------------------------------------------------------
   1 |         1.0000        0.8000 |              1.0000             0.8000
  13 |         0.8844        0.9200 |              1.0000             0.9333
 101 |         0.8311        0.8533 |              1.0000             0.9067
--------------------------------------------------------------------------
Ket luan voi k = 101: Trong so khoang cach giup Accuracy tap kiem tra
tang tu 0.8533 len 0.9067, giup mo hinh bot cau tha ro ret.


In [11]:
# -*- coding: utf-8 -*-
"""Bài tập 5: Chọn k cho KNeighborsRegressor bằng kiểm định chéo trên dữ liệu giá nhà."""

import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.neighbors import KNeighborsRegressor

df = pd.read_csv("data/gia_nha.csv")
X = df[["dien_tich"]]
y = df["gia"]

# Chia dữ liệu giống hệt mục 6 (test_size=0.2, random_state=42)
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Kiem dinh cheo 5 phan tren tap hoc (k tu 1 toi 20):")
diem_am = {}
for k in range(1, 21):
    mh = KNeighborsRegressor(n_neighbors=k)
    # Điểm trả về là neg_mean_absolute_error (mang dấu âm)
    diem_am[k] = cross_val_score(
        mh, X_tr, y_tr, cv=5, scoring="neg_mean_absolute_error"
    ).mean()

# Chọn k có điểm neg_MAE lớn nhất (tức MAE dương nhỏ nhất)
k_tot = max(diem_am, key=diem_am.get)

for k in range(1, 21):
    mae_cv = -diem_am[k]
    dau = " <- tot nhat" if k == k_tot else ""
    print(f"  k = {k:2d}   neg_MAE = {diem_am[k]:7.4f}   (MAE = {mae_cv:.4f}){dau}")
print()

# Chấm trên tập kiểm tra với k tốt nhất và so với đường thẳng
knn_tot = KNeighborsRegressor(n_neighbors=k_tot).fit(X_tr, y_tr)
knn_5 = KNeighborsRegressor(n_neighbors=5).fit(X_tr, y_tr)
duong = LinearRegression().fit(X_tr, y_tr)

mae_knn_tot = mean_absolute_error(y_te, knn_tot.predict(X_te))
mae_knn_5 = mean_absolute_error(y_te, knn_5.predict(X_te))
mae_duong = mean_absolute_error(y_te, duong.predict(X_te))

print(f"k tot nhat theo kiem dinh cheo = {k_tot} (CV MAE = {-diem_am[k_tot]:.4f} ty dong)")
print("So sanh MAE tren tap kiem tra (ty dong) - cang nho cang tot:")
print(f"  KNN voi k = {k_tot:2d} (tot nhat CV): {mae_knn_tot:.4f}")
print(f"  KNN voi k =  5 (muc 6)       : {mae_knn_5:.4f}")
print(f"  Duong thang (LinearReg)      : {mae_duong:.4f}")
print("Ket luan: MAE cua KNN van KHONG vuot duoc duong thang (0.4867 > 0.2578).")

Kiem dinh cheo 5 phan tren tap hoc (k tu 1 toi 20):
  k =  1   neg_MAE = -0.4164   (MAE = 0.4164)
  k =  2   neg_MAE = -0.3498   (MAE = 0.3498) <- tot nhat
  k =  3   neg_MAE = -0.3714   (MAE = 0.3714)
  k =  4   neg_MAE = -0.3618   (MAE = 0.3618)
  k =  5   neg_MAE = -0.3816   (MAE = 0.3816)
  k =  6   neg_MAE = -0.4002   (MAE = 0.4002)
  k =  7   neg_MAE = -0.3990   (MAE = 0.3990)
  k =  8   neg_MAE = -0.4098   (MAE = 0.4098)
  k =  9   neg_MAE = -0.4171   (MAE = 0.4171)
  k = 10   neg_MAE = -0.4365   (MAE = 0.4365)
  k = 11   neg_MAE = -0.4500   (MAE = 0.4500)
  k = 12   neg_MAE = -0.4695   (MAE = 0.4695)
  k = 13   neg_MAE = -0.4992   (MAE = 0.4992)
  k = 14   neg_MAE = -0.5153   (MAE = 0.5153)
  k = 15   neg_MAE = -0.5290   (MAE = 0.5290)
  k = 16   neg_MAE = -0.5595   (MAE = 0.5595)
  k = 17   neg_MAE = -0.5748   (MAE = 0.5748)
  k = 18   neg_MAE = -0.5891   (MAE = 0.5891)
  k = 19   neg_MAE = -0.6130   (MAE = 0.6130)
  k = 20   neg_MAE = -0.6254   (MAE = 0.6254)

k tot nhat theo

In [12]:
# -*- coding: utf-8 -*-
"""Bài tập 6: Đo thời gian huấn luyện (.fit) và dự đoán (.predict) của KNN."""

import time
import numpy as np
from sklearn.neighbors import KNeighborsClassifier

rng = np.random.default_rng(0)

# Tạo sẵn 1000 mẫu mới cần dự đoán với 2 đặc trưng
X_moi = rng.random((1000, 2))
cac_co_du_lieu = [1000, 10000, 100000]

print("Do thoi gian .fit va .predict (cho 1000 mau moi) theo co du lieu:")
print("-" * 62)
print(f"{'So mau (N)':>12} | {'Thoi gian .fit (giay)':>22} | {'Thoi gian .predict (giay)':>22}")
print("-" * 62)

# Dùng algorithm='brute' để thể hiện rõ bản chất tính khoảng cách tới toàn bộ mẫu cũ
# (hoặc để mặc định của KNeighborsClassifier đều thấy .predict chậm hơn .fit rất nhiều)
for n in cac_co_du_lieu:
    X_hoc = rng.random((n, 2))
    y_hoc = rng.integers(0, 2, size=n)

    mo_hinh = KNeighborsClassifier(n_neighbors=5, algorithm="brute")

    # Đo thời gian .fit
    bat_dau_fit = time.perf_counter()
    mo_hinh.fit(X_hoc, y_hoc)
    thoi_gian_fit = time.perf_counter() - bat_dau_fit

    # Đo thời gian .predict cho 1000 mẫu mới
    bat_dau_pred = time.perf_counter()
    mo_hinh.predict(X_moi)
    thoi_gian_pred = time.perf_counter() - bat_dau_pred

    print(f"{n:12d} | {thoi_gian_fit:22.6f} | {thoi_gian_pred:22.6f}")

print("-" * 62)
print("Nhan xet: Thoi gian .fit rat nhanh (gan nhu tuc thi), trong khi thoi gian")
print(".predict cham hon nhieu va tang manh khi so luong mau du lieu lon dan.")

Do thoi gian .fit va .predict (cho 1000 mau moi) theo co du lieu:
--------------------------------------------------------------
  So mau (N) |  Thoi gian .fit (giay) | Thoi gian .predict (giay)
--------------------------------------------------------------
        1000 |               0.001357 |               0.011357
       10000 |               0.001610 |               0.095528
      100000 |               0.012366 |               0.850235
--------------------------------------------------------------
Nhan xet: Thoi gian .fit rat nhanh (gan nhu tuc thi), trong khi thoi gian
.predict cham hon nhieu va tang manh khi so luong mau du lieu lon dan.
